# Lab 1.1 — Exploring Real-World Data
**Module I · LLMs & GNNs for Advanced Reasoning over Relational Data**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_labs/blob/main/module-1-ml/lab1_1_data_exploration.ipynb)

---

## What you will do
Before training any model, you need to *understand* your data. In this lab you will:
1. Load a real business dataset and inspect its structure.
2. Identify and handle data quality issues (missing values, wrong types).
3. Visualise the **class imbalance** — a critical issue that will motivate better evaluation metrics in Lab 1.2.
4. Explore which features seem most related to the outcome you want to predict.

## Dataset: IBM Telco Customer Churn
A telecommunications company wants to know which customers are likely to **churn** (cancel their subscription). The dataset has ~7 000 rows and 21 columns — a mix of numerical and categorical features. This scenario (predicting customer behaviour from relational business data) will be the running example throughout Module I.

## Learning objectives
- Practice pandas for data loading, inspection, and cleaning.
- Recognise class imbalance and understand why it matters.
- Build intuition for which features carry signal before touching any ML.

**Estimated time:** 45–60 min

---
## 0 · Setup
Run the cell below once. It detects whether you are on **Google Colab** (clones the repo) or running **locally** (adjusts the path). Either way, the rest of the notebook works identically.

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/DanielFPerez/llm-gnns-course_labs.git",
         "labs_assignment"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         "labs_assignment/environment/requirements.txt"],
        check=True,
    )
    sys.path.insert(0, "labs_assignment")
else:
    # Running locally from inside module-1-ml/ — go one level up
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from utils import load_telco_churn, plot_class_distribution
from utils.checks import check_dataframe

# Display settings
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams["figure.dpi"] = 110
print("Imports OK.")

---
## 1 · Load the data

In [ ]:
df = load_telco_churn()

### Exercise 1.1 `[Core]` — Initial inspection

Get an overview of the dataset:
- How many rows and columns does it have?
- What are the data types of each column? How many are numerical vs. categorical?
- What do the first few rows look like?

In [ ]:
# YOUR CODE HERE
# Hint: Get an overview of the dataset:
raise NotImplementedError("Complete this exercise")

In [ ]:
df.head()

In [ ]:
check_dataframe("1.1", df, min_rows=7000, required_cols=["customerID", "Churn", "tenure"])

> **What to notice:**
> - There are only 3 numeric columns (`SeniorCitizen`, `tenure`, `MonthlyCharges`). Almost everything else is text.
> - `TotalCharges` is stored as `object` (string), not a number — that is a data quality issue we will fix next.
> - `df.isnull().sum()` reports 0 nulls... but that does not mean the data is clean. Hidden nulls can disguise themselves as empty strings or spaces.

---
## 2 · Cleaning the data

### Exercise 1.2 `[Core]` — Fix `TotalCharges` and encode the target

**Step A — `TotalCharges`:** Convert it from string to float. Use `pd.to_numeric(..., errors='coerce')` which turns any non-numeric entry into `NaN`. Then fill those `NaN`s with `0.0` (they correspond to brand-new customers with 0 tenure).

**Step B — `Churn`:** Convert `"Yes"` → `1` and `"No"` → `0` so the target is a binary integer.

Store the result in a new DataFrame called `df_clean`.

In [ ]:
# YOUR CODE HERE
# Hint: Step A — TotalCharges: Convert it from string to float. Use pd.to_numeric(..., errors='coerce') which turns any non-n...
raise NotImplementedError("Complete this exercise")

In [ ]:
check_dataframe(
    "1.2", df_clean,
    min_rows=7000,
    required_cols=["TotalCharges", "Churn"],
    null_free_cols=["TotalCharges", "Churn"],
)

---
## 3 · Class distribution — the imbalance problem

### Exercise 1.3 `[Core]` — Count churners and non-churners

How many customers churned? What is the ratio of non-churners to churners?

> **Why this matters:** If 80% of customers did *not* churn, a model that always predicts "No Churn" would achieve 80% accuracy — without learning anything useful. This is the *accuracy paradox*, covered in the lecture (Module I, Slide 17). We will revisit it in Lab 1.2 when we choose better metrics.

In [ ]:
# YOUR CODE HERE
# Hint: How many customers churned? What is the ratio of non-churners to churners?
raise NotImplementedError("Complete this exercise")

In [ ]:
plot_class_distribution(
    df_clean["Churn"],
    labels=["No Churn (0)", "Churn (1)"],
    title="Target distribution — Telco Churn",
);

> **Observation:** ~73% of customers did not churn. This is a **moderately imbalanced** dataset. A naïve classifier that always predicts "No Churn" would score **73% accuracy** — yet it would completely fail to identify any actual churners, which is exactly what the business cares about. In Lab 1.2 we will use **precision, recall, and F1** instead.

---
## 4 · Feature exploration

### Exercise 1.4 `[Core]` — Tenure distribution by churn status

`tenure` is the number of months a customer has been with the company. Plot a histogram of `tenure` separately for churners and non-churners (overlapping, with transparency).

> **What to expect:** Customers who churn tend to leave early. If you see that churners cluster at low tenure values, that confirms tenure is a useful feature.

In [ ]:
# YOUR CODE HERE
# Hint: tenure is the number of months a customer has been with the company. Plot a histogram of tenure separately for churne...
raise NotImplementedError("Complete this exercise")

> **Key insight:** Churners are concentrated at low tenure (months 0–12). Long-tenured customers almost never churn. This single feature already tells a rich story and will be one of the most predictive signals for our model.

### Exercise 1.5 `[Core]` — Monthly charges by churn status

Do churners pay more per month? Plot side-by-side **boxplots** of `MonthlyCharges` for churners vs. non-churners.

In [ ]:
# YOUR CODE HERE
# Hint: Do churners pay more per month? Plot side-by-side boxplots of MonthlyCharges for churners vs. non-churners.
raise NotImplementedError("Complete this exercise")

> **Key insight:** Churners tend to have **higher monthly charges**. The median for churners is ~$79 vs. ~$65 for non-churners. This makes intuitive business sense: customers on expensive plans with lower perceived value are more likely to leave.

---
## 5 · `[Extension]` Correlation analysis

This section is optional — tackle it if you finish the core exercises early.

### Exercise 1.6 `[Extension]` — Which features correlate most with churn?

Before training a model, it is useful to know which columns have the strongest linear relationship with the target. Steps:
1. Temporarily encode all categorical columns as integers.
2. Compute the correlation of every feature with `Churn`.
3. Print the top 5 features by absolute correlation.
4. Visualise the correlations for the top 10 features as a heatmap.

In [ ]:
# YOUR CODE HERE
# Hint: Before training a model, it is useful to know which columns have the strongest linear relationship with the target. S...
raise NotImplementedError("Complete this exercise")

In [ ]:
# Heatmap of the top 10 features + target
top_cols = corr_with_churn.head(10).index.tolist() + ["Churn"]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    df_corr[top_cols].corr(),
    annot=True, fmt=".2f", cmap="coolwarm",
    center=0, linewidths=0.5, ax=ax,
)
ax.set_title("Correlation matrix — top 10 features + Churn")
plt.tight_layout()
plt.show()

> **Key insights:**
> - **`Contract`** (month-to-month vs. long-term) is the single feature most correlated with churn.
> - **`tenure`** and **`MonthlyCharges`** confirm what we saw visually above.
> - Some features are highly correlated *with each other* (e.g., `OnlineSecurity` and `TechSupport`) — be aware of this when interpreting model weights later.

---
## Summary

| What we did | Why it matters |
|---|---|
| Loaded a raw CSV and inspected types | Real data always needs inspection before modelling |
| Found a hidden data quality issue in `TotalCharges` | Pandas' `dtype` can hide non-numeric strings |
| Measured the class imbalance (~73 / 27) | Accuracy will be misleading — we need F1 and recall |
| Visualised tenure and charges by churn group | Feature exploration guides model interpretation |
| Computed correlations `[Extension]` | Helps prioritise features before training |

**Next step → Lab 1.2:** We will encode the categorical features, split the data, train a first ML model, and use the right metrics to evaluate it.